# Frozen-B targeted replay poisoning comparison

This exploratory Kaggle study compares **clean**, **20% targeted-poisoned**, and **20% targeted-poisoned-plus-filtered** replay under the already selected clean configuration-B protocol. It reuses the verified five-class Task 1 best checkpoint and frozen continual preprocessing; it does not retrain Task 1 and must not run locally. All historical artifacts stay separate and unchanged.

The uploaded retention evidence selected B with combined accuracy `0.9769527915097729`, macro-F1 `0.670591558187877`, Benign FPR `0.010824470651101`, old accuracy `0.9874783015309689`, and new accuracy `0.02350982350982351`; Infiltration recall was `0.0`. Both A and B completed four epochs and selected epoch one. This is **improved retention with weak acquisition**, not successful eight-class continual learning.

## Frozen protocol declared before this run

The runner reconstructs the exact saved 25,000-row training-only replay (5,000 distinct rows per old class), verifies its original-row IDs and numeric hashes, and saves a private numeric NPZ for reuse. It then changes exactly 4,000 of 20,000 eligible original replay attack labels (IDs 1–4) to Benign: 20% of eligible rows and 16% of all replay rows.

Every arm independently loads the same Task 1 checkpoint and performs the same seeded eight-class expansion. The already-trained B checkpoint is never loaded. Architecture, frozen preprocessing, seed 42, learning rate 0.0001, batch size 256, 70,576 draws per epoch, original B class quotas, maximum eight epochs, patience three, and validation-macro-F1 checkpoint selection are fixed. Sampling always follows the supplied replay labels. The filtered arm stops before training if any required supplied-label class bucket is empty.

The gate uses the frozen Task 1 teacher and the previously verified 95th-percentile clean-Task1-validation threshold. Original labels and poison masks enter only the post-decision audit. Test rows do not tune the attack, gate, training, or checkpoint choice. Each validation-selected arm checkpoint is evaluated once on fixed test rows after all checkpoint hashes are locked.

## 1. Pin the published source and run dependency-light checks

Enable **GPU T4 x2** (or one T4) and **Internet** in Kaggle. This notebook checks out the source commit published before the notebook was created. It retains Kaggle's CUDA-enabled PyTorch and runs focused protocol, replay, poisoning, gate, sampler, metric, and durable checkpoint/resume tests.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, re, shutil, subprocess, sys
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '489c27f85460b193a8ac8b9aa25d9e706f004b97'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-task2-b-poisoning')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import torch
import matplotlib
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable a Kaggle GPU and restart the session')
print({'source_sha': checked_out_sha, 'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'gpu_count': torch.cuda.device_count()})
subprocess.run([sys.executable, '-m', 'py_compile', 'scripts/run_task2_b_poisoning.py'], cwd=REPO_DIR, check=True)
test_specs = [
    ('tests/scripts', 'test_run_task2_b_poisoning.py'),
    ('tests', 'test_poisoning.py'),
    ('tests', 'test_label_consistency.py'),
    ('tests', 'test_disk_replay.py'),
    ('tests', 'test_task2_replay.py'),
    ('tests', 'test_full_disk_training.py'),
    ('tests', 'test_streaming_evaluation.py'),
]
for start_dir, pattern in test_specs:
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', start_dir, '-p', pattern, '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
    print(result.stdout, end=''); print(result.stderr, end='')
    summary = result.stdout + result.stderr
    count = re.search(r'Ran (\d+) tests? in', summary)
    if count is None or int(count.group(1)) == 0 or re.search(r'OK \(skipped=\d+\)', summary):
        raise RuntimeError(f'Focused tests absent or skipped: {start_dir}/{pattern}')

## 2. Select the four read-only Kaggle inputs

Attach these as **private Kaggle datasets**:

1. Complete prepared-data folder `review2_large_prepare_<id>` containing all fixed `.npy` arrays and manifests.
2. Complete Task 1 folder `full_task1_20261003T074354_763068Z` containing `best.pt`, manifests, frozen continual preprocessor, and Task1-before test metrics.
3. Original clean-retention `results.zip`, SHA-256 `1efb433f2f77501a290600a67a7f4e0b1f760b0f5b42b983db49ed1ffd2cacf0`.
4. Original prior full-Task2 `results.zip`, SHA-256 `0c73a44cca06cb92f7c21c98652808f739df67236d58d790f5cf4b93027d832e`, used only to verify the frozen clean-validation calibration record.

The runner rejects hash/path/provenance mismatches and reads only named JSON evidence from structurally safe ZIPs. It does not execute archived code or load the trained B checkpoint.

In [ ]:
PREPARED_DIR = None  # e.g. '/kaggle/input/review2-prepared/review2_large_prepare_20261003T...'
TASK1_RUN_DIR = None  # e.g. '/kaggle/input/review2-task1/full_task1_20261003T074354_763068Z'
RETENTION_RESULTS_ZIP = None  # optional explicit path; otherwise selected by exact hash
PRIOR_TASK2_RESULTS_ZIP = None  # optional explicit path; otherwise selected by exact hash
OUTPUT_DIR = Path('/kaggle/working/task2_b_poisoning_study')
RESUME_RUN_DIR = None  # optional complete prior output under /kaggle/input or /kaggle/working
MAX_EPOCHS_THIS_CALL = None  # optional positive epoch-boundary pause; omit for normal run
EXPECTED_RETENTION_SHA = '1efb433f2f77501a290600a67a7f4e0b1f760b0f5b42b983db49ed1ffd2cacf0'
EXPECTED_PRIOR_SHA = '0c73a44cca06cb92f7c21c98652808f739df67236d58d790f5cf4b93027d832e'
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
def choose_folder(explicit, prefix, marker):
    if explicit is not None:
        folder = Path(explicit)
    else:
        candidates = [p.parent for p in Path('/kaggle/input').rglob(marker) if prefix in p.parent.name]
        print(prefix, 'candidates:', [str(path) for path in candidates])
        if len(candidates) != 1:
            raise RuntimeError(f'Found {len(candidates)} {prefix} folders; set the selector explicitly')
        folder = candidates[0]
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    return folder
def choose_zip(explicit, expected_sha, label):
    candidates = [Path(explicit)] if explicit is not None else list(Path('/kaggle/input').rglob('*.zip'))
    matches = [path for path in candidates if path.is_file() and sha256_file(path) == expected_sha]
    print(label, 'hash matches:', [str(path) for path in matches])
    if len(matches) != 1:
        raise RuntimeError(f'Found {len(matches)} exact {label} ZIPs; set its path explicitly')
    return matches[0]
prepared_dir = choose_folder(PREPARED_DIR, 'review2_large_prepare_', 'features_manifest.json')
task1_dir = choose_folder(TASK1_RUN_DIR, 'full_task1_', 'final_manifest.json')
retention_zip = choose_zip(RETENTION_RESULTS_ZIP, EXPECTED_RETENTION_SHA, 'retention evidence')
prior_task2_zip = choose_zip(PRIOR_TASK2_RESULTS_ZIP, EXPECTED_PRIOR_SHA, 'prior Task2 evidence')
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    if previous.resolve().is_relative_to(Path('/kaggle/working')):
        OUTPUT_DIR = previous
    else:
        if OUTPUT_DIR.exists():
            raise FileExistsError(f'Refusing to overwrite {OUTPUT_DIR}')
        shutil.copytree(previous, OUTPUT_DIR)
print({'prepared_dir': str(prepared_dir), 'task1_dir': str(task1_dir), 'retention_zip': str(retention_zip), 'prior_task2_zip': str(prior_task2_zip), 'output_dir': str(OUTPUT_DIR)})

## 3. Run or resume the three-arm comparison

The runner first verifies evidence, replay provenance, one-pass preprocessing, the exact 4,000-row attack budget, teacher/calibration identity, gate inputs, retained class buckets, B quotas, and identical expansion hashes. It then trains the three arms sequentially. `latest.pt` is authoritative at completed epoch boundaries, while `best.pt` is selected by full clean-validation macro-F1. If a bounded call pauses, preserve the entire output folder and attach/copy it for resume.

Do not infer equal work: the final records report each arm's actual completed epochs, optimizer steps, sampled class exposure, and unique rows. No success threshold is guaranteed.

In [ ]:
command = [
    sys.executable, str(REPO_DIR / 'scripts/run_task2_b_poisoning.py'),
    '--prepared-dir', str(prepared_dir),
    '--task1-run-dir', str(task1_dir),
    '--retention-results-zip', str(retention_zip),
    '--prior-task2-results-zip', str(prior_task2_zip),
    '--output-dir', str(OUTPUT_DIR),
]
if MAX_EPOCHS_THIS_CALL is not None:
    command.extend(['--max-epochs-this-call', str(MAX_EPOCHS_THIS_CALL)])
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

## 4. Review and preserve every output

The final display reads saved JSON only. Preserve the complete output folder as a private Kaggle dataset/version. Required evidence includes protocol/environment manifests; replay, attack, calibration, and gate verification; the private numeric replay/audit NPZs; per-arm configs, histories, latest/best checkpoints, reload checks, validation/test metrics, training curves, normalized confusion matrices and per-class plots; the durable test-open lock; clean-B reproducibility deltas; `comparison.json`; and `study_manifest.json`.

Interpret combined accuracy with old/new accuracy, forgetting, Benign FPR, old-attack-to-Benign errors, per-class recall/F1, and gate poison/clean rejection counts. A clean-control discrepancy is evidence to report, not a reason to tune against test results.

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'study_manifest.json').read_text())
comparison = json.loads((OUTPUT_DIR / 'comparison.json').read_text())
replay = json.loads((OUTPUT_DIR / 'replay_verification.json').read_text())
attack = json.loads((OUTPUT_DIR / 'attack_audit.json').read_text())
gate = json.loads((OUTPUT_DIR / 'gate_audit.json').read_text())
lock = json.loads((OUTPUT_DIR / 'test_open_lock.json').read_text())
assert manifest['source_sha'] == comparison['source_sha'] == lock['source_sha'] == SOURCE_SHA
assert replay['rows'] == replay['unique_original_rows'] == 25000
assert replay['saved_manifest_original_ids_match'] and replay['single_frozen_preprocessing_pass_verified']
assert attack['eligible_original_attack_rows'] == 20000 and attack['changed_rows'] == 4000
assert attack['eligible_fraction'] == 0.20 and attack['total_replay_fraction'] == 0.16
assert gate['ground_truth_used_for_filtering'] is False and gate['ground_truth_used_for_post_gate_audit_only'] is True
assert set(comparison['arms']) == {'clean_B', 'targeted20_B', 'targeted20_filtered_B'}
private_replay = OUTPUT_DIR / replay['private_npz']
assert private_replay.is_file() and sha256_file(private_replay) == replay['private_npz_sha256']
print('Clean-control reproducibility:', json.dumps(comparison['clean_reproducibility'], indent=2))
print('Attack budget:', json.dumps(comparison['attack_budget'], indent=2))
print('Gate poison/clean rejection counts:', json.dumps(comparison['gate_tradeoff'], indent=2))
for name, result in comparison['arms'].items():
    print('\n', name, json.dumps({key: result[key] for key in ('completed_epoch', 'best_epoch', 'optimizer_steps', 'combined_accuracy', 'combined_macro_f1', 'old_accuracy', 'new_accuracy', 'benign_fpr', 'old_attack_to_benign', 'forgetting')}, indent=2))
    print('Per-class recall/F1:', json.dumps(result['per_class_recall_f1'], indent=2))
print('Save the complete private output folder:', OUTPUT_DIR)